In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default ="notebook"

In [ ]:
# df_t1=pd.read_csv('source/uber-raw-data-apr14.csv')
# df_t2=pd.read_csv('source/uber-raw-data-may14.csv')

# df_t3=pd.read_csv('source/uber-raw-data-jun14.csv')

# df_t4=pd.read_csv('source/uber-raw-data-jul14.csv')

# df_t5=pd.read_csv('source/uber-raw-data-aug14.csv')

# df_t6=pd.read_csv('source/uber-raw-data-sep14.csv')

# df_t_all=pd.concat([df_t1,df_t2,df_t3,df_t4,df_t5,df_t6])


In [ ]:
# df_t_all.to_csv('source/uber-raw-data-may-septe-14.csv',index=False)

In [6]:
df_t_all=pd.read_csv('source/uber-raw-data-may-septe-14.csv')

In [3]:
df_t_all.head()

,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [25]:
df_sample=df_t_all.sample(24000,random_state=1)

In [ ]:
df_sample['Date/Time']=pd.to_datetime(df_sample['Date/Time'],format='%m/%d/%Y %H:%M:%S')
df_sample['DayOfWeekNum']=df_sample['Date/Time'].dt.dayofweek
df_sample['DayOfWeek']=df_sample['Date/Time'].dt.day_name()
df_sample['MonthDayNum']=df_sample['Date/Time'].dt.day
df_sample['HourOfDay']=df_sample['Date/Time'].dt.hour



In [27]:
df_sample['Lat']=df_sample['Lat'].astype(float)
df_sample['Lon']=df_sample['Lon'].astype(float)

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans,DBSCAN


sc=StandardScaler()
X_kmean=df_sample[['Lat','Lon']]
X_kmean=sc.fit_transform(X_kmean)




In [14]:
wcss =  []
k = []
for i in range (1,20):
    kmeans = KMeans(n_clusters= i, random_state = 0, n_init = 'auto')
    kmeans.fit(X_kmean)
    wcss.append(kmeans.inertia_)
    k.append(i)

wcss_frame = pd.DataFrame(wcss)
k_frame = pd.Series(k)


fig= px.line(
    wcss_frame,
    x=k_frame,
    y=wcss_frame.iloc[:,-1]
)
fig.update_layout(
    yaxis_title="Inertia",
    xaxis_title="# Clusters",
    title="Inertia per cluster"
)
fig.show() 

In [15]:
# Import silhouette score
from sklearn.metrics import silhouette_score


sil = []
k = []

for i in range (2,20):
    kmeans = KMeans(n_clusters= i, random_state = 0, n_init = 'auto')
    kmeans.fit(X_kmean)
    sil.append(silhouette_score(X_kmean, kmeans.predict(X_kmean)))
    k.append(i)


cluster_scores=pd.DataFrame(sil)
k_frame = pd.Series(k)

fig = px.bar(data_frame=cluster_scores,
             x=k,
             y=cluster_scores.iloc[:, -1]
            )

fig.update_layout(
    yaxis_title="Silhouette Score",
    xaxis_title="# Clusters",
    title="Silhouette Score per cluster"
)

fig.show() 

In [29]:


kmeans=KMeans(n_clusters=6,random_state=0).fit(X_kmean)

df_sample['KMeansCluster']=kmeans.labels_
df_sample['KMeansCluster']=df_sample['KMeansCluster'].astype(str)


In [20]:
pio.renderers.default ="browser"

In [30]:
df_sample.sort_values(by='HourOfDay',inplace=True)

fig=px.scatter_mapbox(df_sample,lat='Lat',lon='Lon',color='KMeansCluster',zoom=9,animation_frame='HourOfDay',title='Uber Trips')
fig.update_layout(mapbox_style='open-street-map')
fig.show()


### DBSCAN

### Diviser mon Dataset en sous dataset

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default ="notebook"

dataset=pd.read_csv('source/uber-raw-data-may-septe-14.csv')
dataset['Date/Time']=pd.to_datetime(dataset['Date/Time'],format='%m/%d/%Y %H:%M:%S')
dataset['HourOfDay']=dataset['Date/Time'].dt.hour



In [5]:
def create_hourly_dataset (df):
    dataset=[]
    for hour in np.arange(0,24):
        df_hour=df.loc[df['HourOfDay']==hour,:]
        df_hour_grouped=df_hour.groupby(['Lat','Lon']).size().reset_index(name='count')
        df_hour_grouped['hour']=hour
        dataset.append(df_hour_grouped)
    return dataset



In [63]:

list_df_hourly=create_hourly_dataset(dataset.sample(648000,random_state=1))

In [64]:
mean_len=[]
for d in list_df_hourly:
    mean_len.append(len(d))
mean_len=np.mean(mean_len)
mean_len

np.float64(22403.458333333332)

In [77]:
fig=px.scatter_mapbox(dataset.sample(100000,random_state=1).sort_values(by='HourOfDay'),lat='Lat',lon='Lon',animation_frame='HourOfDay',zoom=11,title='Uber Trips',mapbox_style='open-street-map',width=1000,height=800)
fig.update_layout(mapbox_style='open-street-map',margin={"r":0,"l":0,})
fig.update_traces(marker_size=3)
fig.show()

In [45]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np

def fit_cluster(df, eps, min_samples):
    X = df[['Lat', 'Lon']]
    # sc = StandardScaler()
    # X = sc.fit_transform(X)

    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean', algorithm='brute')
    dbscan.fit(X)

    return dbscan


In [ ]:
pio.renderers.default ="notebook"

# sample=df_d_h[int(np.random.randint(0,6,1))][int(np.random.randint(0,24,1))]
sample=df_d_h[12]

dbscan_test=fit_cluster(sample,0.006,10)
df=pd.DataFrame(dbscan_test.labels_,columns=['cluster'])
dbscan_test.labels_=dbscan_test.labels_.astype(str)
sample['cluster']=dbscan_test.labels_
sample['cluster']=sample['cluster'].astype(str)
# sample=sample.loc[sample['cluster']!='-1',:]

fig=px.scatter_mapbox(sample,lat='Lat',lon='Lon',color='cluster',zoom=10,mapbox_style='open-street-map')
fig.update_traces(marker_opacity=1,marker_size=4)
fig.update_layout(width=1000,height=500,margin=dict(l=0, r=0, t=0, b=0))
fig.show()


In [46]:
def add_cluster_df(df, eps, min_samples):
    dbscan = fit_cluster(df, eps, min_samples)
    df['cluster'] = dbscan.labels_
    df=df.loc[df['cluster']!=-1,:]
    # df['cluster'] = df['cluster'].astype(str)
    return df

def add_cluster_dataset(dataset, eps, min_samples):
    new_dataset = []
    for df in dataset:
        new_dataset.append(add_cluster_df(df, eps, min_samples))
    return new_dataset

# Exemple d'utilisation

# Supposons que df_d_h soit un dataset valide



In [30]:
df_test=add_cluster_df(df_d_h[17],0.005,100)

fig=px.scatter_mapbox(df_test,lat='Lat',lon='Lon',color='cluster',zoom=10,mapbox_style='open-street-map')
fig.update_traces(marker_opacity=1,marker_size=4)
fig.update_layout(width=1000,height=500,margin=dict(l=0, r=0, t=0, b=0))
fig.show()

In [47]:
from sklearn.cluster import KMeans

import plotly.graph_objects as go

# Function to perform KMeans clustering
def fit_kmeans(df, n_clusters):
    X = df[['Lat', 'Lon']]
    kmeans = KMeans(n_clusters=n_clusters, random_state=0)
    kmeans.fit(X)
    df['cluster_kmeans'] = kmeans.labels_
    return df

# Function to perform DBSCAN clustering
def fit_dbscan(df, eps, min_samples):
    X = df[['Lat', 'Lon']]
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean', algorithm='brute')
    dbscan.fit(X)
    df['cluster_dbscan'] = dbscan.labels_
    return df

# Apply KMeans and DBSCAN clustering to the dataset
# df_d_h_cluster = add_cluster_dataset(df_d_h, 0.005, 100)
df_d_h_cluster = [fit_kmeans(df, 6) for df in df_d_h]
df_d_h_cluster = [fit_dbscan(df, 0.005, 100) for df in df_d_h]

# Function to create the map with a range slider and buttons
def create_map_with_rangeslider_and_buttons(dataset):
    frames = []

    for hour in range(24):
        df = dataset[hour]
        frame = go.Frame(data=[
            go.Scattermapbox(
                lat=df['Lat'],
                lon=df['Lon'],
                mode='markers',
                marker=dict(size=4, color=df['cluster']),
                hoverinfo='text',
                hovertext=df['cluster'],
            )
        ], name=str(hour))
        frames.append(frame)

    fig = go.Figure(
        data=[
            go.Scattermapbox(
                lat=dataset[0]['Lat'],
                lon=dataset[0]['Lon'],
                mode='markers',
                marker=dict(size=4, color=dataset[0]['cluster']),
                hoverinfo='text',
                hovertext=dataset[0]['cluster'],
            )
        ],
        layout=go.Layout(
            width=1000,
            height=800,
            legend=dict(
                title='Clusters',
                orientation='h',
                x=0.5,
                xanchor='center',
                yanchor='top'
            ),
            mapbox=dict(
                center=dict(lat=40.730610, lon=-73.935242),
                style='open-street-map',
                zoom=10,
            ),
            updatemenus=[
                dict(
                    type='buttons',
                    buttons=[

                        dict(label='KMeans',
                             method='update',
                             args=[{'marker.color': [df['cluster_kmeans'] for df in dataset]}]
                             ),
                        dict(label='DBSCAN',
                             method='update',
                             args=[{'marker.color': [df['cluster_dbscan'] for df in dataset]}]
                             )
                    ]
                )
            ],
            sliders=[dict(
                steps=[dict(method='animate',
                            args=[[str(hour)], dict(mode='immediate', frame=dict(duration=500, redraw=True), transition=dict(duration=0))],
                            label=str(hour)) for hour in range(24)],
                transition=dict(duration=0),
                x=0.1,
                xanchor='left',
                y=0,
                yanchor='top'
            )]
        ),
        frames=frames
    )

    return fig

# Create the map
fig = create_map_with_rangeslider_and_buttons(df_d_h_cluster)

# Show the map
fig.show()

KeyError: 'cluster'

In [57]:
from sklearn.cluster import KMeans, DBSCAN
import pandas as pd



# Fonction pour appliquer KMeans
def fit_kmeans(df, n_clusters):
    X = df[['Lat', 'Lon']]
    kmeans = KMeans(n_clusters=n_clusters, random_state=0)
    kmeans.fit(X)
    df['cluster_kmeans'] = kmeans.labels_
    return df

# Fonction pour appliquer DBSCAN
def fit_dbscan(df, eps, min_samples):
    X = df[['Lat', 'Lon']]
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean', algorithm='brute')
    dbscan.fit(X)
    df['cluster_dbscan'] = dbscan.labels_
    return df

# Appliquer KMeans et DBSCAN à chaque sous-dataset pour chaque heure
df_d_h_cluster = [
    [fit_kmeans(df.copy(), 6) for df in df_d_h],
    [fit_dbscan(df.copy(), 0.005, 100) for df in df_d_h]
]

In [58]:
df_d_h_cluster[1][23]

,Lat,Lon,count,hour,cluster_kmeans,cluster_dbscan
0,39.9947,-74.0634,1,23,5,-1
1,40.0161,-74.0571,1,23,5,-1
2,40.1110,-74.0342,2,23,5,-1
3,40.1115,-74.0341,1,23,5,-1
4,40.1522,-74.0329,1,23,5,-1
...,...,...,...,...,...,...
19893,41.0769,-73.8646,1,23,2,-1
19894,41.0806,-73.4708,1,23,1,-1
19895,41.0917,-73.9235,1,23,2,-1
19896,41.0973,-74.1590,1,23,2,-1


In [59]:
import plotly.graph_objects as go

# Fonction pour créer le graphique
def create_map_with_rangeslider_and_buttons(dataset):
    frames = []

    for hour in range(24):
        df_filtered_kmeans = dataset[0][hour].loc[dataset[0][hour]['cluster_kmeans'] != -1]
        kmeans_frame = go.Frame(data=[
            go.Scattermapbox(
                lat=df_filtered_kmeans[0][hour]['Lat'],
                lon=df_filtered_kmeans[0][hour]['Lon'],
                mode='markers',
                marker=dict(size=4, color=df_filtered_kmeans[0][hour]['cluster_kmeans']),
                hoverinfo='text',
                hovertext=df_filtered_kmeans[0][hour]['cluster_kmeans'],
            )
        ], name=f'kmeans_{hour}')
        df_filtered_dbscan = dataset[1][hour].loc[dataset[1][hour]['cluster_kmeans'] != -1]
        dbscan_frame = go.Frame(data=[
            go.Scattermapbox(
                lat=df_filtered_dbscan[1][hour]['Lat'],
                lon=df_filtered_dbscan[1][hour]['Lon'],
                mode='markers',
                marker=dict(size=4, color=df_filtered_dbscan[1][hour]['cluster_dbscan']),
                hoverinfo='text',
                hovertext=df_filtered_dbscan[1][hour]['cluster_dbscan'],
            )
        ], name=f'dbscan_{hour}')
        
        frames.append(kmeans_frame)
        frames.append(dbscan_frame)

    fig = go.Figure(
        data=[
            go.Scattermapbox(
                lat=dataset[0][0]['Lat'],
                lon=dataset[0][0]['Lon'],
                mode='markers',
                marker=dict(size=4, color=dataset[0][0]['cluster_kmeans']),
                hoverinfo='text',
                hovertext=dataset[0][0]['cluster_kmeans'],
            )
        ],
        layout=go.Layout(
            width=1000,
            height=800,
            legend=dict(
                title='Clusters',
                orientation='h',
                x=0.5,
                xanchor='center',
                yanchor='top'
            ),
            mapbox=dict(
                center=dict(lat=40.730610, lon=-73.935242),
                style='open-street-map',
                zoom=10,
            ),
            updatemenus=[
                dict(
                    type='buttons',
                    buttons=[
                        dict(label='KMeans',
                             method='animate',
                             args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True, mode='immediate')]
                             ),
                        dict(label='DBSCAN',
                             method='animate',
                             args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True, mode='immediate')]
                             )
                    ]
                )
            ],
            sliders=[dict(
                steps=[dict(method='animate',
                            args=[[f'kmeans_{hour}', f'dbscan_{hour}'], dict(mode='immediate', frame=dict(duration=500, redraw=True), transition=dict(duration=0))],
                            label=str(hour)) for hour in range(24)],
                transition=dict(duration=0),
                x=0.1,
                xanchor='left',
                y=0,
                yanchor='top'
            )]
        ),
        frames=frames
    )

    return fig

# Créer le graphique
fig = create_map_with_rangeslider_and_buttons(df_d_h_cluster)

# Afficher le graphique
fig.show()

KeyError: 0